In [1]:
import jax 
import jax.numpy as jnp

import haiku as hk

In [69]:
class IntegrateFireNeuron(hk.Module):
    def __init__(self, name="if_neuron", v_init=hk.initializers.Constant(0.0), num_steps=20):
        self.v_init = v_init
        self.num_steps = num_steps
        super().__init__(name=name)
    
    def __call__(self, x):
        # Parameters
        tau_m = hk.get_parameter("tau_m", shape=[], dtype=jnp.float32, init=hk.initializers.Constant(10.)) # Membrane time constant
        v_th = hk.get_parameter("v_th", shape=[], dtype=jnp.float32, init=hk.initializers.Constant(2.0)) # Threshold
        v_reset = hk.get_parameter("v_reset", shape=[], dtype=jnp.float32, init=jnp.zeros) # Reset
        
        v0 = self.v_init(x.shape[1:], dtype=x.dtype)
        spike0 = v0 > v_th
       
        def update_fn(carry, x):
            v0, spike0 = carry
            v = v0 + (x - v0) / tau_m
            spike = v > v_th
            v = jnp.where(spike, v_reset, v)
            return (v, spike), v
        
        # Run update function
        state = x
        init_carry = (v0, spike0)
        _, state= jax.lax.scan(update_fn, init_carry,state)
        
        return state

In [72]:
@hk.without_apply_rng   
@hk.transform
def model_fn(x):
    neuron = IntegrateFireNeuron()
    out = neuron(x)
    print(out)
    return out

params = model_fn.init(jax.random.PRNGKey(42), 0.1*jnp.ones((20,1)))

[[ 1.]
 [-8.]
 [ 0.]
 [ 1.]
 [-8.]
 [ 0.]
 [ 1.]
 [-8.]
 [ 0.]
 [ 1.]
 [-8.]
 [ 0.]
 [ 1.]
 [-8.]
 [ 0.]
 [ 1.]
 [-8.]
 [ 0.]
 [ 1.]
 [-8.]]
